# 🤖 Osztályozás a gépi tanulásban

Ez a notebook **lépésről lépésre** vezet végig az osztályozás (classification) témakörén.  
Minden cellát futtass le sorban — a magyarázatok és a kód együtt tanítanak.

---

## Mit fogsz megtanulni?

| # | Témakör |
|---|---|
| 1 | Mi az osztályozás és mikor használjuk? |
| 2 | Adatkészlet betöltése és megismerése |
| 3 | Vizualizáció — az adatok megértése |
| 4 | Adatok előkészítése (train/test split, normalizálás) |
| 5 | Logisztikus regresszió |
| 6 | Döntési fa (Decision Tree) |
| 7 | Véletlen erdő (Random Forest) |
| 8 | K-Legközelebbi Szomszéd (KNN) |
| 9 | Modellek összehasonlítása |
| 10 | Konfúziós mátrix és metrikák |

---
**Adatkészlet:** Iris — 150 virág, 3 faj, 4 jellemző  
**Cél:** megtanuljuk melyik virág melyik fajhoz tartozik a méreteik alapján.

## 1. Mi az osztályozás?

Az **osztályozás** (classification) olyan gépi tanulási feladat, ahol egy bemeneti adathoz **kategóriát** rendelünk.

### Példák a való életből:
- 📧 **Spam szűrő** → az e-mail spam vagy nem spam?
- 🏥 **Diagnózis** → a daganat jóindulatú vagy rosszindulatú?
- 🌸 **Virágfaj** → melyik fajhoz tartozik a virág?
- 💳 **Banki csalás** → a tranzakció csalárd vagy legitim?

### Bináris vs. többosztályos:
- **Bináris** (binary): 2 osztály → igen/nem, spam/nem spam
- **Többosztályos** (multiclass): 3+ osztály → mi lesz a mai notebook témája 🌸

## 2. Könyvtárak importálása

Először importáljuk az összes szükséges Python könyvtárat.

In [ ]:
# Adatkezelés
import numpy as np
import pandas as pd

# Vizualizáció
import matplotlib.pyplot as plt
import seaborn as sns

# Adatkészlet és előfeldolgozás
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Osztályozó algoritmusok
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Kiértékelési metrikák
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Esztétikus ábrák
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid', palette='husl')

print('✅ Minden könyvtár sikeresen betöltve!')

## 3. Az Iris adatkészlet betöltése

Az **Iris adatkészlet** a gépi tanulás "Hello World"-je.  
3 virágfajt tartalmaz, mindegyikből 50 mérés:

| Jellemző | Magyar neve |
|---|---|
| sepal length (cm) | csészelevél hossza |
| sepal width (cm) | csészelevél szélessége |
| petal length (cm) | sziromlevél hossza |
| petal width (cm) | sziromlevél szélessége |

**Célváltozó (amit megjósolunk):** melyik fajhoz tartozik?
- 0 → Iris Setosa
- 1 → Iris Versicolor  
- 2 → Iris Virginica

In [ ]:
# Adatkészlet betöltése
iris = load_iris()

# DataFrame létrehozása — emberi olvasáshoz
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['faj'] = iris.target
df['faj_neve'] = df['faj'].map({0: 'Setosa', 1: 'Versicolor', 2: 'Virginica'})

print(f'Sorok száma: {df.shape[0]}')
print(f'Oszlopok száma: {df.shape[1]}')
print(f'\nFajok megoszlása:')
print(df['faj_neve'].value_counts())

df.head(10)

In [ ]:
# Alapstatisztikák — minden jellemző eloszlása
df.describe().round(2)

## 4. Adatok vizualizálása (EDA)

Mielőtt modellt tanítunk, **meg kell értenünk az adatokat**.  
Nézzük meg, hogyan különülnek el a fajok a jellemzők alapján.

In [ ]:
# Jellemzők eloszlása fajok szerint
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
jellemzők = iris.feature_names
magyar_nevek = ['Csészelevél hossza (cm)', 'Csészelevél szélessége (cm)',
                'Sziromlevél hossza (cm)', 'Sziromlevél szélessége (cm)']

for ax, jellemző, magyar in zip(axes.flat, jellemzők, magyar_nevek):
    for faj in df['faj_neve'].unique():
        adatok = df[df['faj_neve'] == faj][jellemző]
        ax.hist(adatok, alpha=0.6, label=faj, bins=15)
    ax.set_title(magyar, fontsize=12, fontweight='bold')
    ax.set_xlabel('Érték (cm)')
    ax.set_ylabel('Darabszám')
    ax.legend()

plt.suptitle('Jellemzők eloszlása fajok szerint', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 Megfigyelés: a sziromlevél méretei jól elkülönítik a fajokat!')

In [ ]:
# Scatter plot — sziromlevél hossza vs szélessége
plt.figure(figsize=(10, 7))
colors = {'Setosa': '#e74c3c', 'Versicolor': '#2ecc71', 'Virginica': '#3498db'}

for faj, szin in colors.items():
    adatok = df[df['faj_neve'] == faj]
    plt.scatter(
        adatok['petal length (cm)'],
        adatok['petal width (cm)'],
        c=szin, label=faj, s=80, alpha=0.8, edgecolors='white', linewidth=0.5
    )

plt.xlabel('Sziromlevél hossza (cm)', fontsize=13)
plt.ylabel('Sziromlevél szélessége (cm)', fontsize=13)
plt.title('Virágfajok elkülönülése sziromlevél méretei alapján', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.show()

print('💡 A Setosa teljesen elkülönül — a Versicolor és Virginica kicsit átfed.')

In [ ]:
# Korreláció hőtérkép — melyik jellemzők függnek össze?
plt.figure(figsize=(8, 6))
korrelacio = df[iris.feature_names].corr()
sns.heatmap(
    korrelacio,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    xticklabels=['Cs.hossz', 'Cs.széless.', 'Sz.hossz', 'Sz.széless.'],
    yticklabels=['Cs.hossz', 'Cs.széless.', 'Sz.hossz', 'Sz.széless.']
)
plt.title('Jellemzők korrelációja', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 A sziromlevél hossza és szélessége erősen korrelál (0.96) — szinte ugyanazt mérik.')

## 5. Adatok előkészítése

Mielőtt tanítunk egy modellt, el kell készíteni az adatokat:

1. **X / y szétválasztás** — jellemzők vs. célváltozó
2. **Train/test split** — tanító és teszt halmaz szétválasztása
3. **Normalizálás** — jellemzők azonos skálára hozása

### Miért kell train/test split?

Ha ugyanazon adaton teszteljük a modellt, amin tanítottuk → a modell "bemagolta" az adatokat → nem tudja megítélni az **általánosítóképességét**.  
A teszt halmaz szimulál **ismeretlen, új adatokat**.

In [ ]:
# X = jellemzők, y = célváltozó (faj)
X = iris.data
y = iris.target

print(f'X alakja: {X.shape}  → {X.shape[0]} minta, {X.shape[1]} jellemző')
print(f'y alakja: {y.shape}  → {y.shape[0]} cimke')

# 80% tanítás, 20% teszt
# random_state=42 → reprodukálható eredmény (mindig ugyanúgy osztja szét)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTanítóhalmaz mérete: {X_train.shape[0]} minta')
print(f'Teszthalmaz mérete:  {X_test.shape[0]} minta')
print('\n💡 stratify=y → gondoskodik arról, hogy mindkét halmazban egyenlő arányban legyenek a fajok')

In [ ]:
# Normalizálás (StandardScaler)
# Minden jellemzőt átalakít: átlag=0, szórás=1
# Ez fontos a távolságalapú modelleknél (KNN) és a logisztikus regressziónál!

scaler = StandardScaler()

# FONTOS: csak a tanítóhalmazon "tanulunk" (fit), aztán mindkettőre alkalmazzuk (transform)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # ← NEM fit_transform! Csak transform.

print('Normalizálás előtt (első minta):', X_train[0].round(2))
print('Normalizálás után  (első minta):', X_train_scaled[0].round(2))
print('\n💡 Miért csak transform a teszthalmazon? Mert a teszt "ismeretlen" adat —')
print('   ha azt is bevonnánk a skálázásba, info-szivárgás (data leakage) történne!')

## 6. Logisztikus regresszió

Az első algoritmusunk — neve megtévesztő, ez **osztályozásra** való, nem regresszióra!

### Hogyan működik?
Egy S-alakú **szigmoid függvénnyel** számítja ki minden osztályhoz tartozás valószínűségét.  
A legmagasabb valószínűségű osztály lesz a jóslat.

**Erőssége:** egyszerű, gyors, jól értelmezhető  
**Gyengesége:** lineáris határt húz → komplex, nemlineáris adatoknál gyengébb

In [ ]:
# Modell létrehozása és tanítása
log_reg = LogisticRegression(max_iter=200, random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Jóslat a teszthalmazon
y_pred_lr = log_reg.predict(X_test_scaled)

# Pontosság
acc_lr = accuracy_score(y_test, y_pred_lr)
print(f'Logisztikus regresszió pontossága: {acc_lr:.1%}')

# Nézzük meg az első néhány jóslat
print('\nValódi:  ', y_test[:10])
print('Jósolt:  ', y_pred_lr[:10])
print('Egyezik: ', (y_test[:10] == y_pred_lr[:10]).astype(int), '← 1=helyes, 0=hibás')

## 7. Döntési fa (Decision Tree)

A döntési fa **if-else kérdések** sorozata — mint egy folyamatábra.

### Hogyan működik?
```
sziromlevél hossza ≤ 2.45 cm?
├── IGEN → Setosa ✅
└── NEM → sziromlevél szélessége ≤ 1.75 cm?
          ├── IGEN → Versicolor ✅  
          └── NEM  → Virginica ✅
```

**Erőssége:** vizuálisan érthető, nem kell normalizálás  
**Gyengesége:** hajlamos túlilleszkedni (overfitting) — "bemagol"

In [ ]:
# max_depth=3 → legfeljebb 3 kérdés mélységig mehet (korlátozza az overfittinget)
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)  # ← döntési fa nem igényel normalizálást!

y_pred_dt = dt.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)
print(f'Döntési fa pontossága: {acc_dt:.1%}')

In [ ]:
# A fa vizualizálása — ez az egyik legnagyobb előnye!
plt.figure(figsize=(16, 8))
plot_tree(
    dt,
    feature_names=['Cs.hossz', 'Cs.széless.', 'Sz.hossz', 'Sz.széless.'],
    class_names=['Setosa', 'Versicolor', 'Virginica'],
    filled=True,
    rounded=True,
    fontsize=11
)
plt.title('Döntési fa — az osztályozás logikája', fontsize=14, fontweight='bold')
plt.show()

print('💡 Olvasd el a fát felülről lefelé — minden csúcspont egy kérdés!')
print('   A szín a domináns osztályt jelöli az adott ágban.')

## 8. Véletlen erdő (Random Forest)

A véletlen erdő sok döntési fa **együttese** — mindenki szavaz, a többség dönt.

### Hogyan javít a döntési fán?
- **100 különböző fa** tanul az adatok különböző részletein
- Mindegyik kicsit más eredményt ad
- Az átlaguk sokkal stabilabb és pontosabb

Ez az **ensemble learning** lényege: a sok gyenge tanuló együtt erős.

**Erőssége:** kiváló pontosság, robusztus, nem overfittel  
**Gyengesége:** nehezebb értelmezni mint egy fa, lassabb

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print(f'Véletlen erdő pontossága: {acc_rf:.1%}')

# Jellemzők fontossága — melyik változó számít a legtöbbet?
fontossag = pd.Series(rf.feature_importances_, index=iris.feature_names).sort_values(ascending=True)

plt.figure(figsize=(8, 4))
colors = ['#e74c3c' if v == fontossag.max() else '#3498db' for v in fontossag]
fontossag.plot(kind='barh', color=colors)
plt.title('Jellemzők fontossága (Random Forest)', fontsize=13, fontweight='bold')
plt.xlabel('Fontosság (0-1)')
plt.tight_layout()
plt.show()

print('\n💡 A sziromlevél hossza a legfontosabb jellemző — ez látszott már a scatter plotokon is!')

## 9. K-Legközelebbi Szomszéd (KNN)

Az egyik legintuítívabb algoritmus — **szomszédok alapján dönt**.

### Hogyan működik?
1. Jön egy ismeretlen virág
2. Megkeresük a K legközelebbi ismert virágot (távolság alapján)
3. Szavazás: melyik faj van többségben a szomszédok között?
4. Az az osztály lesz a jóslat

**K=5** → az 5 legközelebbi szomszéd szavaz

**Erőssége:** egyszerű, nem igényel tanítási fázist ("lusta tanuló")  
**Gyengesége:** nagy adathalmazon lassú, érzékeny a skálázásra → normalizálás kell!

In [ ]:
# Megkeressük az optimális K értéket
k_ertekek = range(1, 21)
pontossagok = []

for k in k_ertekek:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    pontossag = accuracy_score(y_test, knn.predict(X_test_scaled))
    pontossagok.append(pontossag)

plt.figure(figsize=(10, 5))
plt.plot(k_ertekek, pontossagok, 'bo-', linewidth=2, markersize=8)
legjobb_k = k_ertekek[np.argmax(pontossagok)]
plt.axvline(x=legjobb_k, color='red', linestyle='--', label=f'Legjobb K={legjobb_k}')
plt.xlabel('K értéke (szomszédok száma)', fontsize=12)
plt.ylabel('Pontosság', fontsize=12)
plt.title('KNN pontossága különböző K értékekre', fontsize=13, fontweight='bold')
plt.legend(fontsize=12)
plt.xticks(k_ertekek)
plt.ylim(0.9, 1.02)
plt.show()

print(f'Legjobb K: {legjobb_k}')
print('💡 Kis K → túlilleszkedés (overfitting) | Nagy K → alulilleszkedés (underfitting)')

In [ ]:
# Legjobb KNN modell
knn_best = KNeighborsClassifier(n_neighbors=legjobb_k)
knn_best.fit(X_train_scaled, y_train)

y_pred_knn = knn_best.predict(X_test_scaled)
acc_knn = accuracy_score(y_test, y_pred_knn)
print(f'KNN pontossága (K={legjobb_k}): {acc_knn:.1%}')

## 10. Modellek összehasonlítása

Tegyük egymás mellé az összes modell eredményét!

In [ ]:
# Összefoglaló táblázat
eredmenyek = pd.DataFrame({
    'Modell': ['Logisztikus regresszió', 'Döntési fa', 'Véletlen erdő', f'KNN (K={legjobb_k})'],
    'Pontosság': [acc_lr, acc_dt, acc_rf, acc_knn],
    'Normalizálás kell?': ['Igen', 'Nem', 'Nem', 'Igen'],
    'Értelmezhető?': ['Közepes', 'Nagyon', 'Kevéssé', 'Közepes']
})
eredmenyek['Pontosság (%)'] = (eredmenyek['Pontosság'] * 100).round(1).astype(str) + '%'
eredmenyek = eredmenyek.drop('Pontosság', axis=1)

print('=== MODELLEK ÖSSZEHASONLÍTÁSA ===')
print(eredmenyek.to_string(index=False))

# Vizualizáció
nevek = ['Log. Regresszió', 'Döntési fa', 'Véletlen erdő', f'KNN (K={legjobb_k})']
pontok = [acc_lr, acc_dt, acc_rf, acc_knn]
szinek = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

plt.figure(figsize=(10, 5))
bars = plt.bar(nevek, pontok, color=szinek, edgecolor='white', linewidth=1.5)
for bar, pont in zip(bars, pontok):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{pont:.1%}', ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.ylim(0.85, 1.05)
plt.ylabel('Pontosság', fontsize=12)
plt.title('Osztályozó modellek összehasonlítása', fontsize=14, fontweight='bold')
plt.xticks(fontsize=11)
plt.tight_layout()
plt.show()

## 11. Konfúziós mátrix és értékelési metrikák

A **pontosság** (accuracy) önmagában nem elég — nézzük meg részletesebben!

### Konfúziós mátrix:
Megmutatja, **miket tévesztett** el a modell — melyik osztályt keverte össze melyikkel.

```
              Jósolt
           Set  Ver  Vir
Valódi Set [ TP  FP   FP ]
       Ver [ FN  TP   FP ]
       Vir [ FN  FN   TP ]
```
Az átló = helyes jóslatok ✅  
Az átlón kívül = tévesztések ❌

In [ ]:
# Konfúziós mátrixok mind a 4 modellre
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
modellek = [
    ('Log. Regresszió', y_pred_lr),
    ('Döntési fa', y_pred_dt),
    ('Véletlen erdő', y_pred_rf),
    (f'KNN (K={legjobb_k})', y_pred_knn)
]

for ax, (nev, y_pred) in zip(axes, modellek):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Setosa', 'Versicolor', 'Virginica'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(nev, fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Konfúziós mátrixok', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 Tökéletes modell: csak az átlón vannak számok (minden egyezik).')
print('   Tévesztések: Versicolor ↔ Virginica — ezek a leginkább hasonló fajok.')

In [ ]:
# Részletes riport a legjobb modellre
legjobb_pontossag = max(acc_lr, acc_dt, acc_rf, acc_knn)
legjobb_pred = [y_pred_lr, y_pred_dt, y_pred_rf, y_pred_knn][np.argmax([acc_lr, acc_dt, acc_rf, acc_knn])]
legjobb_nev = ['Logisztikus regresszió', 'Döntési fa', 'Véletlen erdő', f'KNN (K={legjobb_k})'][np.argmax([acc_lr, acc_dt, acc_rf, acc_knn])]

print(f'=== LEGJOBB MODELL: {legjobb_nev} ({legjobb_pontossag:.1%}) ===')
print()
print(classification_report(y_test, legjobb_pred, target_names=['Setosa', 'Versicolor', 'Virginica']))
print()
print('Metrikák magyarázata:')
print('  precision  = TP / (TP + FP) → ha azt mondja "Setosa", hányszor igaz?')
print('  recall     = TP / (TP + FN) → az összes Setosát megtalálta-e?')
print('  f1-score   = a kettő harmonikus átlaga — az egyensúly mérőszáma')
print('  support    = valódi minták száma az osztályban')

## 12. Saját adatra jóslás

Próbáljuk ki a modellt egy teljesen saját méréssel!

In [ ]:
# Kísérletezz! Változtasd meg a méréseket és nézd meg mi változik.
saját_virág = np.array([
    [5.1, 3.5, 1.4, 0.2],   # kis értékek → valószínűleg Setosa
    [6.0, 2.9, 4.5, 1.5],   # közepes → valószínűleg Versicolor
    [6.9, 3.1, 5.4, 2.1],   # nagy értékek → valószínűleg Virginica
])

faj_nevek = {0: '🌸 Setosa', 1: '🌺 Versicolor', 2: '🌷 Virginica'}

saját_scaled = scaler.transform(saját_virág)
joslat = rf.predict(saját_virág)  # Random Forest
valoszinuseg = rf.predict_proba(saját_virág)

print('=== JÓSLAT SAJÁT ADATOKRA (Random Forest) ===')
print()
oszlop_nevek = ['Cs.hossz', 'Cs.széless.', 'Sz.hossz', 'Sz.széless.']
for i, (virág, j, prob) in enumerate(zip(saját_virág, joslat, valoszinuseg)):
    mérések = ', '.join(f'{n}={v}' for n, v in zip(oszlop_nevek, virág))
    print(f'Virág {i+1}: [{mérések}]')
    print(f'  → Jóslat: {faj_nevek[j]}')
    print(f'  → Valószínűségek: Setosa={prob[0]:.1%}, Versicolor={prob[1]:.1%}, Virginica={prob[2]:.1%}')
    print()

## 13. Összefoglalás

### Mit tanultunk?

| Lépés | Mit csináltunk? |
|---|---|
| EDA | Az adatok vizualizálásával megértettük az összefüggéseket |
| Train/test split | Szétválasztottuk a tanítási és tesztelési adatokat |
| Normalizálás | Az értékeket azonos skálára hoztuk |
| Modell tanítás | 4 különböző algoritmus megtanulta az adatokat |
| Kiértékelés | Pontosság, konfúziós mátrix, precision/recall/F1 |

### Melyik modellt válasszam?

| Helyzet | Ajánlott modell |
|---|---|
| Gyors alapvonal, érthető eredmény | Logisztikus regresszió |
| Érthetőség a legfontosabb | Döntési fa |
| Legjobb pontosság kell | Véletlen erdő |
| Kis adat, egyszerű feladat | KNN |

### Következő lépések 🚀
- **Keresztvalidáció** (cross-validation) — megbízhatóbb értékelés
- **Hiperparaméter hangolás** (GridSearchCV) — az optimális beállítások keresése
- **SVM** és **Gradient Boosting** (XGBoost) — erős alternatívák
- **Valódi adatkészletek** — Titanic, MNIST, saját adataid

---

> **Ez a notebook Claude Code segítségével készült, mobilról — a `claude/mobile-dev-capabilities-fnhgd6` branchen.**